# Train a New Policy Using ONLY the Time2Success Model as Reward

Pure model-derived reward — no original dense reward, no ground-truth
success bonus, nothing else mixed in. This is deliberately the "pure env"
variant only; the other reward variants explored earlier (additive shaping,
diff+bonus) are intentionally left out of this notebook, matching the
explicit scope requested.

**Honesty check to keep in mind throughout:** the policy gets zero
ground-truth signal during training under this setup — `eval/success_rate`
in the callback output (computed on the real env, real success flag) is the
ONLY place reality re-enters. Watch it closely, especially during the smoke test.

## 1. Setup — env, constants, and the model class definition

In [1]:
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import metaworld
import collections
import json, os, time

TASK_NAME = "peg-insert-side-v3"
SUCCESS_KEY = "success"

def make_env(seed=0, render_mode=None):
    return gym.make("Meta-World/MT1", env_name=TASK_NAME, seed=seed, render_mode=render_mode)

_probe = make_env(seed=0)
DT = getattr(_probe.unwrapped, "dt", _probe.unwrapped.model.opt.timestep)
OBS_DIM = _probe.observation_space.shape[0]
_probe.close()
print(f"DT={DT}, OBS_DIM={OBS_DIM}")

class Time2SuccessModel(nn.Module):
    def __init__(self, obs_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = "cuda" if torch.cuda.is_available() else "cpu"
# NOTE: no dataset loading, no fresh/untrained model instantiation here —
# that was dead code left over from the training notebook. The only
# Time2SuccessModel instance used below is loaded from the saved checkpoint
# in the next cell.

DT=0.0125, OBS_DIM=39


d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:34: UserWarning: WARN: A Box observation space maximum and minimum values are equal.
  logger.warn("A Box observation space maximum and minimum values are equal.")


## 2. Load the frozen, trained time2success model — explicit loading point

This is the ONLY place the model gets constructed for this notebook's purposes.

In [2]:
norm = np.load("./checkpoints/time2success/normalization.npz")
SHAPING_X_MEAN, SHAPING_X_STD = norm["X_mean"], norm["X_std"]
SHAPING_Y_MEAN, SHAPING_Y_STD = norm["y_mean"].item(), norm["y_std"].item()

shaping_t2s_model = Time2SuccessModel(obs_dim=OBS_DIM).to(device)
shaping_t2s_model.load_state_dict(
    torch.load("./checkpoints/time2success/time2success_state_model_best.pt"))
shaping_t2s_model.eval()  # frozen — strictly no gradients during RL training
print("Loaded frozen time2success model + normalization stats")

Loaded frozen time2success model + normalization stats


## 3. The pure reward wrapper

Only the model-derived time-difference signal. `success_bonus` from the
uploaded version is removed entirely (it was accepted but never used —
dead code); if you want a ground-truth bonus back later, that's a
deliberate, separate variant, not something to silently carry as an unused
parameter.

In [3]:
GAMMA = 0.99

class PureTime2SuccessRewardWrapper(gym.Wrapper):
    def __init__(self, env, t2s_model, device, x_mean, x_std, y_mean, y_std,
                 gamma=GAMMA, shaping_scale=1.0, stall_window=20):
        super().__init__(env)
        self.t2s_model = t2s_model
        self.device = device
        self.x_mean, self.x_std = x_mean, x_std
        self.y_mean, self.y_std = y_mean, y_std
        self.gamma = gamma
        self.shaping_scale = shaping_scale
        self.stall_window = stall_window
        self._last_phi = 0.0
        self._recent_predictions = collections.deque(maxlen=self.stall_window)

    @torch.no_grad()
    def _potential(self, obs):
        x_norm = (obs - self.x_mean) / self.x_std
        x = torch.tensor(x_norm, dtype=torch.float32).unsqueeze(0).to(self.device)
        pred_norm = self.t2s_model(x).item()
        pred_steps = pred_norm * self.y_std + self.y_mean
        return -pred_steps  # higher potential = closer to success

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._last_phi = self._potential(obs)
        self._recent_predictions.clear()
        self._recent_predictions.append(-self._last_phi)
        return obs, info

    def step(self, action):
        obs, _base_reward, terminated, truncated, info = self.env.step(action)  # original reward discarded

        phi_next = self._potential(obs)
        predicted_now = -phi_next

        # Stall detection: cut the rollout short if predicted time2success
        # hasn't improved over the last `stall_window` steps — saves compute
        # on rollouts that are going nowhere.
        if (len(self._recent_predictions) == self.stall_window
                and predicted_now >= max(self._recent_predictions)):
            truncated = True

        self._recent_predictions.append(predicted_now)

        pure_t2s_reward = self.gamma * phi_next - self._last_phi
        self._last_phi = phi_next

        total_reward = self.shaping_scale * pure_t2s_reward
        return obs, total_reward, terminated, truncated, info

def make_pure_t2s_env(seed=0):
    base_env = make_env(seed=seed)
    return PureTime2SuccessRewardWrapper(
        base_env, shaping_t2s_model, device,
        SHAPING_X_MEAN, SHAPING_X_STD, SHAPING_Y_MEAN, SHAPING_Y_STD,
        gamma=GAMMA, shaping_scale=1.0, stall_window=20,
    )

## 4. Build the training setup (consistently named — `pure_*` throughout, nothing else)

In [4]:
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from stable_baselines3.common.callbacks import BaseCallback

N_ENVS = 6
pure_train_env = DummyVecEnv([lambda i=i: make_pure_t2s_env(seed=i) for i in range(N_ENVS)])
pure_train_env = VecMonitor(pure_train_env)
pure_eval_env = make_env(seed=1000)  # eval always on the RAW env/real success — never the shaped reward

pure_model = SAC(
    policy="MlpPolicy",
    env=pure_train_env,
    learning_rate=3e-4,
    buffer_size=1_000_000,
    batch_size=256,
    tau=0.005,
    gamma=GAMMA,
    ent_coef="auto",
    policy_kwargs=dict(net_arch=[400, 400]),
    tensorboard_log="./tb_logs/peg_insert_side_pure_t2s",
    verbose=1,
    seed=0,
)

d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:34: UserWarning: WARN: A Box observation space maximum and minimum values are equal.
  logger.warn("A Box observation space maximum and minimum values are equal.")


Using cpu device


In [10]:
class SuccessCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=10_000, n_eval_episodes=10,
                 ckpt_dir="./checkpoints/peg_insert_side_pure_t2s", ckpt_freq=100_000, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.ckpt_dir = ckpt_dir
        self.ckpt_freq = ckpt_freq
        os.makedirs(ckpt_dir, exist_ok=True)
        self.history = []

    def _run_eval_episode(self):
        obs, _ = self.eval_env.reset()
        success_step = None
        for t in range(500):
            action, _ = self.model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = self.eval_env.step(action)
            if info.get(SUCCESS_KEY, 0) and success_step is None:
                success_step = t
            if terminated or truncated:
                break
        return success_step

    def _on_step(self) -> bool:
        if self.num_timesteps % self.ckpt_freq < self.training_env.num_envs:
            self.model.save(os.path.join(self.ckpt_dir, f"pure_t2s_{self.num_timesteps}.zip"))

        if self.num_timesteps % self.eval_freq < self.training_env.num_envs:
            steps = [self._run_eval_episode() for _ in range(self.n_eval_episodes)]
            success_rate = sum(s is not None for s in steps) / self.n_eval_episodes
            times = [s for s in steps if s is not None]
            mean_t2s = float(np.mean(times)) if times else None
            self.logger.record("eval/success_rate", success_rate)
            if mean_t2s is not None:
                self.logger.record("eval/mean_time_to_success", mean_t2s)
            self.history.append(dict(step=self.num_timesteps, success_rate=success_rate,
                                       mean_time_to_success=mean_t2s, timestamp=time.time()))
            with open(os.path.join(self.ckpt_dir, "eval_history.json"), "w") as f:
                json.dump(self.history, f, indent=2)
            if self.verbose:
                print(f"[eval @ {self.num_timesteps}] success_rate={success_rate:.2f} mean_t2s={mean_t2s}")
        return True

# Correctly bound to the pure setup — this was the actual crash before
# (referenced a nonexistent `shaped_eval_env` from a different variant).
pure_callback = SuccessCallback(eval_env=pure_eval_env)


## 5. Smoke test — run this before the full run

In [8]:
pure_model.learn(total_timesteps=5_000, callback=pure_callback, tb_log_name="pure_t2s_smoke")
print("Smoke test complete — check eval_history.json under ./checkpoints/peg_insert_side_pure_t2s/")
print("Also worth checking: is eval/success_rate doing anything sane, given there\'s no ground-truth reward at all?")

Logging to ./tb_logs/peg_insert_side_pure_t2s\pure_t2s_smoke_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 20       |
|    ep_rew_mean     | 5.26     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 431      |
|    time_elapsed    | 0        |
|    total_timesteps | 120      |
| train/             |          |
|    actor_loss      | -13.5    |
|    critic_loss     | 0.55     |
|    ent_coef        | 0.782    |
|    ent_coef_loss   | -1.66    |
|    learning_rate   | 0.0003   |
|    n_updates       | 821      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.4     |
|    ep_rew_mean     | 6.61     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 315      |
|    time_elapsed    | 0        |
|    total_timesteps | 246      |
| train/             |          |
|    actor_loss    

**Before scaling up**, print a few `total_reward` values from one rollout
and eyeball the scale (not wildly huge/tiny) — `shaping_scale=1.0` is an
untested default, same caveat as always.

## 6. Full run — only after the smoke test looks reasonable

In [ ]:
TOTAL_TIMESTEPS = 3_000_000

pure_model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=pure_callback,
    tb_log_name="pure_t2s_run1",
    progress_bar=True,
    reset_num_timesteps=False,
)
pure_model.save("./checkpoints/peg_insert_side_pure_t2s/sac_peg_insert_pure_t2s_final")
pure_train_env.close()
pure_eval_env.close()
print("Pure time2success-reward training complete")

Logging to ./tb_logs/peg_insert_side_pure_t2s\pure_t2s_run1_0


Output()

## 7. Compare against stage 1

Since this run has zero ground-truth reward during training, this comparison
is the real verdict on whether pure model-derived reward can teach the task
at all.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

with open("./checkpoints/peg_insert_side/eval_history.json") as f:
    orig_history = json.load(f)
with open("./checkpoints/peg_insert_side_pure_t2s/eval_history.json") as f:
    pure_history = json.load(f)

orig_df = pd.DataFrame(orig_history)
pure_df = pd.DataFrame(pure_history)

plt.figure(figsize=(7,4))
plt.plot(orig_df["step"], orig_df["success_rate"], label="original (dense reward)")
plt.plot(pure_df["step"], pure_df["success_rate"], label="pure time2success reward")
plt.xlabel("step")
plt.ylabel("success rate")
plt.legend()
plt.title("Original vs. pure time2success-only reward")
plt.show()

Evaluation

In [1]:
import gymnasium as gym
import metaworld

TASK_NAME = "peg-insert-side-v3"
SUCCESS_KEY = "success"  # match whatever your flag-check found earlier

def make_env(seed=0, render_mode=None):
    env = gym.make(
        "Meta-World/MT1",
        env_name=TASK_NAME,
        seed=seed,
        render_mode=render_mode,
        camera_name="corner3"
    )
    return env

In [8]:
import glob
import os
import numpy as np
import imageio
from stable_baselines3 import SAC

# --- Load the most recent checkpoint from this run ---
ckpt_dir = "./checkpoints/peg_insert_side_pure_t2s"
ckpt_files = glob.glob(os.path.join(ckpt_dir, "pure_t2s_*.zip"))

if not ckpt_files:
    print("No checkpoints found — check ckpt_freq was actually small enough to have saved one by now")
else:
    # sort by the step number embedded in the filename, not alphabetically
    ckpt_files.sort(key=lambda p: int(os.path.basename(p).replace("pure_t2s_", "").replace(".zip", "")))
    latest_ckpt = ckpt_files[-1]
    # latest_ckpt = "./checkpoints/peg_insert_side_pure_t2s/pure_t2s_900000.zip"
    
    print(f"Loading: {latest_ckpt}")

    eval_policy = SAC.load(latest_ckpt)

    def record_rollout(model, env, out_path, max_steps=500, deterministic=True):
        obs, _ = env.reset()
        frames = []
        success_step = None
        for t in range(max_steps):
            frames.append(env.render())
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = env.step(action)
            if info.get(SUCCESS_KEY, 0) and success_step is None:
                success_step = t
            if terminated or truncated:
                break
        imageio.mimsave(out_path, frames, fps=20)
        return success_step, len(frames)

    render_env = make_env(seed=1, render_mode="rgb_array")
    success_step, num_frames = record_rollout(eval_policy, render_env, "pure_t2s_current_behavior.mp4")
    render_env.close()

    if success_step is not None:
        print(f"Succeeded at step {success_step}")
    else:
        print(f"Did not succeed within {num_frames} steps — video still saved, worth watching what it's actually doing")

Loading: ./checkpoints/peg_insert_side_pure_t2s\pure_t2s_2400000.zip
Did not succeed within 500 steps — video still saved, worth watching what it's actually doing
